# Generate Biweekly averages of the `lead` dimension

---

Creates biweekly averages in the `lead` dimension of the raw or anomaly files.

Also at the bottom of the notebook we compute biweekly averages for the verification data as well.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import xarray as xr
xr.set_options(keep_attrs=True)

import dask.config
dask.config.set({"array.slicing.split_large_chunks": False})

In [2]:
from dask_cluster import get_client
client = get_client()

Choose your model and data type

In [17]:
model = "MPAS_60km"
data = "anom" #raw or anom
average = "biweekly"
zarr_dir = "output/"

In [18]:
hinda = xr.open_zarr(f"{zarr_dir}{model}.{data}.daily.geospatial.zarr/", consolidated=True).astype("float32")

Because each model has a different `lead` dimension size I needed to hard code some things in.

In [19]:
res = "RR" if model.count("_") > 1 else "none"
if res=="RR":
    hinda["lead"] = range(0,29) # for MPAS RR
else:
    hinda["lead"] = range(0,46)

A function to calculate the biweekly averages.

In [ ]:
if average == "biweekly":
    def make_average(hind):
        # Standard 14-day windows: days 1-14, 15-28, 29-42 (lead = first day of the window)
        init_w12 = hind.sel(lead=range(1,1+14)).mean(dim='lead', skipna=False)
        init_w34 = hind.sel(lead=range(15,15+14)).mean(dim='lead', skipna=False)
        if res=="RR":
            hind_bw = xr.concat([init_w12,init_w34], dim='lead')
            hind_bw["lead"] = [1,15] # lead represents first day of biweekly aggregate
        else:
            init_w56 = hind.sel(lead=range(29,29+14)).mean(dim='lead', skipna=False)
            hind_bw = xr.concat([init_w12, init_w34, init_w56], dim='lead')
            hind_bw["lead"] = [1,15,29] # lead represents first day of biweekly aggregate
        hind_bw["lead"].attrs["units"] = "days"
        return hind_bw

elif average == "weekly":
    def make_average(hind):
        w1 = hind.sel(lead=range(1,8)).mean(dim="lead",skipna=False)
        w2 = hind.sel(lead=range(8,15)).mean(dim="lead",skipna=False)
        w3 = hind.sel(lead=range(15,22)).mean(dim="lead",skipna=False)
        if res=="RR":
            w4 = hind.sel(lead=range(22,29)).mean(dim="lead",skipna=False)
            hind_w = xr.concat([w1,w2,w3,w4],dim="lead")
            hind_w["lead"] = [1,8,15,22] # lead represents first day of weekly aggregate
        else:
            w4 = hind.sel(lead=range(22,29)).mean(dim="lead",skipna=False)
            w5 = hind.sel(lead=range(29,36)).mean(dim="lead",skipna=False)
            w6 = hind.sel(lead=range(36,43)).mean(dim="lead",skipna=False)
            hind_w = xr.concat([w1,w2,w3,w4,w5,w6],dim="lead")
            hind_w["lead"] = [1,8,15,22,29,36] # lead represents first day of weekly aggregate
        hind_w["lead"].attrs["units"] = "days"
        
        return hind_w

## Apply the function to make biweekly averages

In [21]:
hind_av = make_average(hinda)

Rechunk and write the data to a new `zarr` file!

In [22]:
hind_av = hind_av.chunk({"member": -1,"init": 1,"lead": -1,"lat": 181,"lon": 360}).persist()
hind_av

<xarray.Dataset> Size: 16GB
Dimensions:  (lead: 3, init: 1040, member: 10, lat: 181, lon: 360)
Coordinates:
  * init     (init) datetime64[ns] 8kB 2000-01-06 2000-01-13 ... 2019-12-28
  * lat      (lat) float32 724B -90.0 -89.0 -88.0 -87.0 ... 87.0 88.0 89.0 90.0
  * lon      (lon) float32 1kB 0.0 1.0 2.0 3.0 4.0 ... 356.0 357.0 358.0 359.0
  * member   (member) int64 80B 1 2 3 4 5 6 7 8 9 10
  * lead     (lead) int64 24B 1 15 29
Data variables:
    t2m      (lead, init, member, lat, lon) float32 8GB dask.array<chunksize=(3, 1, 10, 181, 360), meta=np.ndarray>
    tp       (lead, init, member, lat, lon) float32 8GB dask.array<chunksize=(3, 1, 10, 181, 360), meta=np.ndarray>
Attributes:
    Conventions:    CF-1.6
    history:        2025-11-04 04:02:19 GMT by grib_to_netcdf-2.39.4: /opt/ec...
    regrid_method:  bilinear

In [23]:
%time hind_av.to_zarr(f"{zarr_dir}{model}.{data}.{average}.geospatial.zarr/",mode="w",consolidated=True)

CPU times: user 411 ms, sys: 40.3 ms, total: 451 ms
Wall time: 32.8 s


## Biweekly averages of the verification data (ERA5)

Each value at date `t` is the 14-day mean of days `t` through `t+13`. climpred compares a forecast at `lead` with the observation at `init + lead`, so the observation windows line up with the model windows above (lead 1 = days 1-14, lead 15 = days 15-28, lead 29 = days 29-42).

You only need to run this once per `data` type (raw or anom), not once per model. **It overwrites the existing `OBS_ERA5.{data}.biweekly` file.**

In [6]:
if average == "biweekly":
    # ERA5 daily data comes from the data catalog (it was not made by notebooks 1-3)
    import intake
    cat = intake.open_catalog("s2s_data_catalog.yml")
    obs = cat["OBS_ERA5"](data=data, lead="daily", dim="geospatial").to_dask().astype("float32")
    # rolling() puts the 14-day mean on the LAST day of the window, so drop the first 13 values
    # and relabel each mean with the FIRST day of its window.
    obs_av = obs.rolling(time=14).mean().isel(time=slice(13, None))
    obs_av["time"] = obs.time.values[:-13]
    obs_av = obs_av.chunk({"time": 366, "lat": 181, "lon": 360})
    obs_av.to_zarr(f"{zarr_dir}OBS_ERA5.{data}.{average}.geospatial.zarr/", mode="w", consolidated=True)